# B2-023-generative-models-diffusion — Practice p19 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** denoising-diffusion-probabilistic-models

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

## Solution

**Plan.** Re-run p18's pinned training protocol exactly (same split readers, schedule, `q_sample`, `ddpm_loss`, `Denoiser`, seed placement, optimizer, and `train_ddpm` with its internal generator), with the module-level `q_sample` wrapped by a recorder for the duration of the training call only.
Then implement p11's `p_sample`/`sample_loop`, draw the fixed starting noise `x_T` and then the step noises `noises` from one generator seeded `SEED + 2`, and sample 512 points under `torch.no_grad()`.
The samples are compared with the **held-out** rows' mean and covariance, checked for mode coverage (p08 rule, radius 0.6), and for precision (`inside_fraction`) against the starting noise.

In [ ]:
import copy

DATASET = "mixture2d"
T = 50


def train_rows():
    return generative_data.train_tensor(DATASET)


def heldout_rows():
    return generative_data.heldout_tensor(DATASET)


def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def q_sample(x0, t, eps, alpha_bar):
    if x0.shape != eps.shape:
        raise ValueError("x0 and eps must have the same shape")
    if (not isinstance(t, torch.Tensor) or t.dtype != torch.int64 or t.dim() != 1
            or x0.dim() != 2 or t.shape[0] != x0.shape[0]):
        raise ValueError("t must be an int64 tensor of shape (B,)")
    T_max = alpha_bar.shape[0] - 1
    if bool((t < 1).any()) or bool((t > T_max).any()):
        raise ValueError("timesteps must lie in 1..T")
    ab = alpha_bar[t].to(x0.dtype).unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps


def ddpm_loss(model, x0, t, eps, alpha_bar):
    T_max = alpha_bar.shape[0] - 1
    x_t = q_sample(x0, t, eps, alpha_bar)            # module-level name, looked up at call time
    eps_hat = model(x_t, (t.to(x0.dtype) / T_max).unsqueeze(1))
    if eps_hat.shape != eps.shape:
        raise ValueError("model output must have the shape of eps")
    return ((eps_hat - eps) ** 2).mean()


class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, x_t, t_scaled):
        return self.net(torch.cat([x_t, t_scaled], dim=1))


def train_ddpm(model, batch, optimizer):
    noise_generator = torch.Generator().manual_seed(SEED)
    trace = []
    for _ in range(1500):
        t = torch.randint(1, T + 1, (batch.shape[0],), generator=noise_generator)
        eps = torch.randn(batch.shape[0], 2, generator=noise_generator)
        optimizer.zero_grad(set_to_none=True)
        loss = ddpm_loss(model, batch, t, eps, SCHEDULE["alpha_bar"])
        loss.backward()
        optimizer.step()
        trace.append(loss.item())
    return trace


def split_hash_sets(name):
    hashes = generative_data.ROW_SHA256[name]
    return ({hashes[i] for i in generative_data.TRAIN_IDS[name]},
            {hashes[i] for i in generative_data.HELDOUT_IDS[name]})


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)
SCHEDULE = make_schedule(50, 1e-3, 0.2)


def p_sample(model, x_t, t, z, schedule):
    T_max = schedule["beta"].shape[0] - 1
    if not isinstance(t, int) or isinstance(t, bool) or not 1 <= t <= T_max:
        raise ValueError("t must be an int in 1..T")
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T_max, dtype=x_t.dtype))
    beta_t = schedule["beta"][t].item()
    alpha_t = schedule["alpha"][t].item()
    alpha_bar_t = schedule["alpha_bar"][t].item()
    mean = (x_t - beta_t / math.sqrt(1 - alpha_bar_t) * eps_hat) / math.sqrt(alpha_t)
    if t == 1:
        return mean
    return mean + math.sqrt(schedule["beta_tilde"][t].item()) * z


def sample_loop(model, x_T, noises, schedule):
    T_max = schedule["beta"].shape[0] - 1
    if tuple(noises.shape) != (T_max, *x_T.shape):
        raise ValueError("noises must have shape (T, *x_T.shape)")
    x = x_T
    for t in range(T_max, 0, -1):
        x = p_sample(model, x, t, noises[t - 1], schedule)
    return x


def mode_coverage(samples, centers, radius):
    if not isinstance(samples, torch.Tensor) or not isinstance(centers, torch.Tensor):
        raise ValueError("samples and centers must be tensors")
    if samples.dim() != 2 or centers.dim() != 2 or samples.shape[1] != centers.shape[1]:
        raise ValueError("samples and centers must be 2-D with equal last dimensions")
    if samples.shape[0] == 0:
        raise ValueError("need at least one sample")
    if not isinstance(radius, (int, float)) or isinstance(radius, bool) or not radius > 0:
        raise ValueError("radius must be positive")
    d2 = ((samples[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)
    nearest = d2.argmin(dim=1)
    inside = d2.gather(1, nearest[:, None]).squeeze(1) <= radius ** 2
    counts = torch.bincount(nearest[inside], minlength=centers.shape[0]).to(torch.int64)
    need = math.ceil(samples.shape[0] / (8 * centers.shape[0]))
    return counts, int((counts >= need).sum().item())


CENTERS = generative_data.mixture_centers()


def inside_fraction(points, radius=0.6):
    d2 = ((points[:, None, :] - CENTERS[None, :, :]) ** 2).sum(dim=2)
    return (d2.min(dim=1).values <= radius ** 2).float().mean().item()

### Training (p18 protocol) with the seam recorded

In [ ]:
torch.manual_seed(SEED)
model = Denoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)

recorded = []
original_q_sample = q_sample


def recording_q_sample(x0, t, eps, alpha_bar):
    recorded.append(x0.detach().clone())
    return original_q_sample(x0, t, eps, alpha_bar)


q_sample = recording_q_sample
try:
    trace = train_ddpm(model, train_rows(), optimizer)
finally:
    q_sample = original_q_sample
seam_restored = q_sample is original_q_sample
recorded_rows = torch.unique(torch.cat(recorded), dim=0)
recorded_hashes = {generative_data.row_sha256(row) for row in recorded_rows}
first50, last50 = sum(trace[:50]) / 50, sum(trace[-50:]) / 50
print(f"first-50 mean {first50:.4f}  last-50 mean {last50:.4f}  ratio {last50 / first50:.3f}")

### Fixed-noise sampling and the statistics

In [ ]:
sample_generator = torch.Generator().manual_seed(SEED + 2)
x_T = torch.randn(512, 2, generator=sample_generator)
noises = torch.randn(50, 512, 2, generator=sample_generator)
with torch.no_grad():
    samples = sample_loop(model, x_T, noises, SCHEDULE)
    samples_again = sample_loop(model, x_T, noises, SCHEDULE)

H = heldout_rows()
mean_gap = (samples.mean(dim=0) - H.mean(dim=0)).abs()
cov_gap = (torch.cov(samples.T) - torch.cov(H.T)).abs()
counts, covered = mode_coverage(samples, CENTERS, 0.6)
frac_samples, frac_noise = inside_fraction(samples), inside_fraction(x_T)
noise_counts, noise_covered = mode_coverage(x_T, CENTERS, 0.6)
print("sample mean", [round(v, 3) for v in samples.mean(dim=0).tolist()], " held-out mean", [round(v, 3) for v in H.mean(dim=0).tolist()])
print("sample cov", [round(v, 3) for v in torch.cov(samples.T).flatten().tolist()], " held-out cov", [round(v, 3) for v in torch.cov(H.T).flatten().tolist()])
print("max |mean gap|", round(mean_gap.max().item(), 4), " max |cov gap|", round(cov_gap.max().item(), 4))
print("coverage counts", counts.tolist(), " covered", covered)
print("starting-noise coverage counts", noise_counts.tolist(), " covered", noise_covered)
print(f"inside fraction: samples {frac_samples:.3f}  starting noise {frac_noise:.3f}")

### Interpretation

(D) alone cannot distinguish samples from pure noise, because 512 standard-normal points already put more than 16 points within $0.6$ of each of the four centres, whereas (E) measures the **fraction** of all samples that land inside a radius, which the starting noise fails ($\le0.5$) and the sampler passes ($\ge0.75$).
The comparison uses held-out statistics because training rows were the targets the network was fitted to, so matching them could reflect memorization, while matching rows the model never saw tests the distribution it learned.
With only 64 held-out rows, each held-out mean coordinate has a standard error of about $\sqrt{1.04/64}\approx0.13$ and each covariance entry also fluctuates by roughly $0.1$–$0.2$, so tolerances of $0.35$ and $0.30$ (two to three standard errors) are the tightest that a correct sampler passes reliably.
A sampler that added noise at its final step would give every final sample an extra random displacement that no later step removes, spreading each blob outward and lowering the inside fraction in (E) (with this schedule's $\beta_1=0.001$ the displacement has standard deviation only about $0.03$, but any larger final noise erodes precision visibly), while (D)'s coverage would barely move.

### Answer check

In [ ]:
assert train_rows().shape == (256, 2) and H.shape == (64, 2)
# training decrease
assert len(trace) == 1500 and all(math.isfinite(v) for v in trace)
assert last50 <= 0.60 * first50
# (A)
assert tuple(samples.shape) == (512, 2) and samples.dtype == torch.float32 and bool(torch.isfinite(samples).all())
# (B)
assert bool((mean_gap <= 0.35).all())
# (C)
assert bool((cov_gap <= 0.30).all())
# (D)
assert covered == 4
assert noise_covered == 4          # (D) alone does not separate samples from noise
# (E)
assert frac_samples >= 0.75 and frac_noise <= 0.5
# (F)
assert torch.equal(samples, samples_again)
# (G)
assert seam_restored and len(recorded) == 1500
assert not (recorded_hashes & HELDOUT_HASHES)
assert TRAIN_HASHES <= recorded_hashes